In [10]:
# pip install nfl_data_py pandas pyarrow

import pandas as pd
from nfl_data_py import import_pbp_data

# 1) Fetch 2023 PBP (avoid downcasting so columns like wpa aren't dropped)
pbp = import_pbp_data([2023], downcast=False, cache=False)

# 2) Ensure WPA/WP_POST exist (recompute if wpa wasn't included)
if 'wpa' not in pbp.columns:
    # next play's pre-play WP within the same game
    pbp['wp_post'] = pbp.groupby('game_id')['wp'].shift(-1)
    pbp['wpa'] = pbp['wp_post'] - pbp['wp']
else:
    # provide wp_post for convenience
    pbp['wp_post'] = pbp['wp'] + pbp['wpa']

# 3) Keep a tidy subset of useful columns
keep = [
    'game_id','play_id','season','week','posteam','defteam','home_team','away_team',
    'down','ydstogo','yardline_100','qtr','half_seconds_remaining',
    'score_differential','posteam_timeouts_remaining','defteam_timeouts_remaining',
    'roof','surface','temp','wind','play_type','field_goal_result','punt_blocked',
    'wp','wpa','wp_post','ep','epa'
]
pbp = pbp[[c for c in keep if c in pbp.columns]].copy()

# 4) Optional: just 4th-down decision states
pbp_4th = pbp[pbp['down'] == 4].copy()

print(pbp_4th[['play_id','down','ydstogo','yardline_100','wp','wpa','wp_post']].head())
# pbp_4th.to_parquet("pbp_4th_2023.parquet", index=False)

2023 done.
    play_id  down  ydstogo  yardline_100        wp       wpa   wp_post
10    245.0   4.0      7.0          49.0  0.496160  0.021550  0.517710
17    427.0   4.0      8.0          70.0  0.432813  0.038686  0.471498
36    958.0   4.0      9.0          11.0  0.313076  0.032166  0.345242
46   1237.0   4.0      9.0          36.0  0.383763  0.015041  0.398805
60   1601.0   4.0     11.0          85.0  0.390952 -0.041274  0.349678


In [11]:
pbp_4th

,game_id,play_id,season,week,posteam,defteam,home_team,away_team,down,ydstogo,...,temp,wind,play_type,field_goal_result,punt_blocked,wp,wpa,wp_post,ep,epa
10,2023_01_ARI_WAS,245.0,2023,1,WAS,ARI,WAS,ARI,4.0,7.0,...,NaN,NaN,punt,None,0.0,0.496160,0.021550,0.517710,0.144536,-0.415890
17,2023_01_ARI_WAS,427.0,2023,1,ARI,WAS,WAS,ARI,4.0,8.0,...,NaN,NaN,punt,None,0.0,0.432813,0.038686,0.471498,-1.407519,1.075530
36,2023_01_ARI_WAS,958.0,2023,1,ARI,WAS,WAS,ARI,4.0,9.0,...,NaN,NaN,field_goal,made,0.0,0.313076,0.032166,0.345242,2.874272,0.125728
46,2023_01_ARI_WAS,1237.0,2023,1,ARI,WAS,WAS,ARI,4.0,9.0,...,NaN,NaN,field_goal,made,0.0,0.383763,0.015041,0.398805,0.901252,2.098748
60,2023_01_ARI_WAS,1601.0,2023,1,ARI,WAS,WAS,ARI,4.0,11.0,...,NaN,NaN,punt,None,0.0,0.390952,-0.041274,0.349678,-1.932299,-0.807140
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49589,2023_22_SF_KC,3156.0,2023,22,SF,KC,KC,SF,4.0,3.0,...,NaN,NaN,pass,None,0.0,0.444412,0.101789,0.546201,2.782424,2.397201
49605,2023_22_SF_KC,3534.0,2023,22,KC,SF,KC,SF,4.0,6.0,...,NaN,NaN,field_goal,made,0.0,0.529621,-0.003928,0.525693,2.969886,0.030114
49613,2023_22_SF_KC,3733.0,2023,22,SF,KC,KC,SF,4.0,5.0,...,NaN,NaN,field_goal,made,0.0,0.533539,0.120908,0.654447,1.442558,1.557442
49647,2023_22_SF_KC,4525.0,2023,22,SF,KC,KC,SF,4.0,4.0,...,NaN,NaN,field_goal,made,0.0,0.988725,-0.514933,0.473792,2.935563,0.064437


In [12]:
pbp

,game_id,play_id,season,week,posteam,defteam,home_team,away_team,down,ydstogo,...,temp,wind,play_type,field_goal_result,punt_blocked,wp,wpa,wp_post,ep,epa
0,2023_01_ARI_WAS,1.0,2023,1,None,None,WAS,ARI,NaN,0.0,...,NaN,NaN,None,None,NaN,0.546262,0.000000,0.546262,1.474098,0.000000
1,2023_01_ARI_WAS,39.0,2023,1,WAS,ARI,WAS,ARI,NaN,0.0,...,NaN,NaN,kickoff,None,0.0,0.546262,0.000000,0.546262,1.474098,0.000000
2,2023_01_ARI_WAS,55.0,2023,1,WAS,ARI,WAS,ARI,1.0,10.0,...,NaN,NaN,run,None,0.0,0.546262,-0.006641,0.539621,1.474098,-0.336103
3,2023_01_ARI_WAS,77.0,2023,1,WAS,ARI,WAS,ARI,2.0,7.0,...,NaN,NaN,pass,None,0.0,0.539621,0.016367,0.555987,1.137994,0.703308
4,2023_01_ARI_WAS,102.0,2023,1,WAS,ARI,WAS,ARI,3.0,1.0,...,NaN,NaN,run,None,0.0,0.555987,0.016586,0.572573,1.841302,0.469799
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49660,2023_22_SF_KC,4791.0,2023,22,KC,SF,KC,SF,3.0,1.0,...,NaN,NaN,run,None,0.0,0.648316,0.209521,0.857837,3.136118,1.728891
49661,2023_22_SF_KC,4813.0,2023,22,KC,SF,KC,SF,1.0,10.0,...,NaN,NaN,run,None,0.0,0.857837,-0.007961,0.849875,4.865008,-0.330699
49662,2023_22_SF_KC,4835.0,2023,22,KC,SF,KC,SF,2.0,7.0,...,NaN,NaN,pass,None,0.0,0.849875,0.021764,0.871640,4.534309,-0.637115
49663,2023_22_SF_KC,4860.0,2023,22,KC,SF,KC,SF,1.0,3.0,...,NaN,NaN,pass,None,0.0,0.871640,0.128360,1.000000,3.897194,3.102806


In [14]:
pbp.columns.tolist()

['game_id',
 'play_id',
 'season',
 'week',
 'posteam',
 'defteam',
 'home_team',
 'away_team',
 'down',
 'ydstogo',
 'yardline_100',
 'qtr',
 'half_seconds_remaining',
 'score_differential',
 'posteam_timeouts_remaining',
 'defteam_timeouts_remaining',
 'roof',
 'surface',
 'temp',
 'wind',
 'play_type',
 'field_goal_result',
 'punt_blocked',
 'wp',
 'wpa',
 'wp_post',
 'ep',
 'epa']